In [ ]:
import spikeinterface.full as si
import numpy as np
from scipy.io import savemat
import pickle
import ks_util as ksu

import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.qualitymetrics as sqm
import spikeinterface.exporters as sexp

import matplotlib.pyplot as plt

# from kilosort.io import save_probe
import os
from pathlib import Path
import gc
import json

from spikeinterface.sortingcomponents.peak_detection import detect_peaks
from spikeinterface.sortingcomponents.peak_localization import localize_peaks
from spikeinterface.sortingcomponents.motion import interpolate_motion,estimate_motion,correct_motion_on_peaks
import matplotlib.pyplot as plt

In [ ]:
basepaths =  [ Path( r"D:\acquisition\IND65_100625" )]

for basepath in basepaths:

    basename = os.path.basename( basepath )
    # identify recording files- NOTE, there might be more than one !
    files = [f for f in os.listdir( basepath ) if f.startswith('202')]
    oe_path = Path.joinpath( basepath, files[0] )
    
    this_node = '101'

    # paths to OE experiments- in my case different probe maps, so I want to sort these separately 
    exp_paths = ksu.get_experiment_paths( oe_path, node=this_node  )
    
    # this will not work if we have more than one experiment ! 
    this_expPath = exp_paths[ 0 ]
    
    # set paths
    neural_dat = ksu.dat_path_oe( this_expPath, node=this_node )
    adc_dat = ksu.adc_path_oe( this_expPath, node=this_node  )
    preprocessing_path = Path.joinpath( basepath, 'preprocessing' )
    
    # load and save metadata for analogin datastream 
    oe_recording_analogin = se.read_openephys( this_expPath / 'recording1', stream_id='2' )
    # adc_metadata_folder = Path.joinpath( basepath, 'analogin_metadata' )
    # adc_metadata_folder.mkdir( exist_ok=True )
    # oe_recording_analogin.save_metadata_to_folder( adc_metadata_folder )
    # # save paths to analog data as a json file
    # with open( Path.joinpath( adc_metadata_folder, 'datapaths.json' ), "w" ) as f:
    #     json.dump([str(p) for p in adc_dat], f, indent=2)
    
    # load and save metadata for neural datastream 
    oe_recording_neural = se.read_openephys(this_expPath / 'recording1', stream_id='1') # read intan?
    # neural_metadata_folder = Path.joinpath( basepath, 'neural_metadata' )
    # neural_metadata_folder.mkdir(exist_ok=True)
    # oe_recording_neural.save_metadata_to_folder( neural_metadata_folder )
    # with open( Path.joinpath( neural_metadata_folder, 'datapaths.json' ), "w" ) as f:
    #     json.dump([str(p) for p in neural_dat], f, indent=2)
    
    nchan = oe_recording_neural.get_num_channels()
    fs = oe_recording_neural.get_sampling_frequency()
    # define a binary recording extractor  
    binary_recording = se.BinaryRecordingExtractor(
                            file_paths = neural_dat[0],
                            sampling_frequency = fs,
                            num_channels = nchan,
                            dtype = "int16",
                            time_axis = 0,
                        )
    
    # load the .dat file via 
    binary_recording.load_metadata_from_folder( neural_metadata_folder )
    
    # load metadata into a data frame and save a chanMap.mat file
    # df = binary_recording.get_probe().to_dataframe()
    # chanMap = {
    #     'chanMap0ind': np.arange( 0, nchan ),
    #     'chanMap': np.arange( 1, nchan+1 ),
    #     'xcoords': np.array( df['x'] ),
    #     'ycoords': np.array( df['y'] ),
    #     'kcoords': np.array( df['shank_ids'] ).astype( np.int32 )+1,
    #     'n_chan': nchan,
    #     'connected': np.full( nchan, True )
    # }
    
    # savemat( basepath / 'chanMap.mat', chanMap)

    # phase shift 
    # binary_recording_ps = si.phase_shift( binary_recording ) 
    # # save phase shifted as a separate file
    # binary_recording_ps_ready = binary_recording_ps.save( folder=preprocessing_path )
    
    # binary_recording_final = se.BinaryRecordingExtractor(
    #                         file_paths = preprocessing_path / 'traces_cached_seg0.raw',
    #                         sampling_frequency = fs,
    #                         num_channels = nchan,
    #                         dtype = "int16",
    #                         time_axis = 0,
    #                     )
    # load the .dat file via 
    # binary_recording_final.load_metadata_from_folder( neural_metadata_folder )
    
    outfolder=basepath / Path( 'Kilosort4_' + ksu.get_date() )
    # run kilosort
    sorting_ks4 = ss.run_sorter(sorter_name="kilosort4", recording=binary_recording_final, folder=outfolder, torch_device='cuda', verbose=True, remove_existing_folder=True)

    del binary_recording, binary_recording_final, binary_recording_ps_ready, binary_recording_ps, sorting_ks4
    gc.collect()
    
    with open( basepath / 'A1_done_sorting.txt', "a") as f:
        f.write("...")